# 复现 Fig. 6-7：每晕直接捕获率的红移演化

这个 Notebook 沿着暗物质晕的平均质量吸积史计算两体直接捕获率：

1. 先由今天质量 $M_0$ 得到 $M(z)$；
2. `twobodycapture.capture_rate_per_halo_a14_per_year` 在每个 $(M(z),z)$ 上重新计算浓度、NFW 结构、速度平均与质量积分；
3. Fig. 6 比较一个 $M_0=10^{12}M_\odot$ 晕中的单色谱和 log-normal 捕获率，并叠加 $M(z)$；
4. Fig. 7 比较五个不同 $M_0$ 晕的单色谱捕获率。

物理公式仍只保存在主体 `.py` 模块中，本 Notebook 负责图像专用参数、循环、绘图和 CSV。

## 1. 导入模块并定位项目目录

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "twobodycapture.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import haloconcentration as hc
import pbhmassfunction as pmf
import twobodycapture as capture

PROJECT_ROOT

## 2. 设置公共红移网格与 PBH 质量函数

红移数组包含 $z=0$ 和 49 个正红移点，范围到 $z=12$，与此前 Fig. 9 使用的时间采样一致。$z=0$ 会写入 CSV，但不能画在对数横轴上。

论文 Fig. 6 使用 $m=30M_\odot$ 单色谱，以及 $M_c=30M_\odot,\sigma=0.6$ 的 log-normal 分布；Fig. 7 只使用同一单色谱。

In [ ]:
REDSHIFT = np.linspace(0.0, 12.0, 50)
POSITIVE_REDSHIFT = REDSHIFT > 0.0
F_PBH = 1.0

monochromatic_nodes, monochromatic_weights = (
    pmf.monochromatic_mass_distribution(mass_msun=30.0)
)

def lognormal_sigma_0p6_pdf(mass_msun):
    return pmf.lognormal_mass_pdf(
        mass_msun,
        median_mass_msun=30.0,
        sigma=0.6,
    )

lognormal_nodes, lognormal_weights = pmf.continuous_mass_quadrature(
    lognormal_sigma_0p6_pdf,
    point_count=96,
)

REDSHIFT[[0, 1, 12, 24, 36, 49]]

## 3. 定义沿质量吸积史计算捕获率的 Notebook 辅助函数

输入是今天质量 $M_0$ 和 PBH 质量积分测度。函数先调用 Ludlow16 的质量吸积史，再逐个红移调用 Appendix A(14) 每晕捕获率。

这里必须把同一行始终视为同一个 $M_0$ 晕的历史，不能在不同红移处重新按质量排序。

In [ ]:
def capture_rate_along_halo_history(
    present_day_mass_msun,
    mass_nodes_msun,
    mass_probability_weights,
):
    mass_track_msun = hc.mass_accretion_history(
        present_day_mass_msun,
        REDSHIFT,
        model="ludlow16",
    )

    rate_track_per_year = np.array([
        capture.capture_rate_per_halo_a14_per_year(
            halo_mass_msun,
            z,
            mass_nodes_msun=mass_nodes_msun,
            mass_probability_weights=mass_probability_weights,
            concentration_model="ludlow16",
            f_pbh=F_PBH,
        )
        for halo_mass_msun, z in zip(mass_track_msun, REDSHIFT)
    ])

    return mass_track_msun, rate_track_per_year

# Fig. 6：$M_0=10^{12}M_\odot$ 晕的捕获率和质量演化

## 4. 计算 Fig. 6 的质量、单色谱率和 log-normal 率

两种 PBH 质量函数使用相同的 $M(z)$、晕浓度与速度环境，区别只来自 Appendix A(14) 的双质量积分。

In [ ]:
FIG6_PRESENT_DAY_MASS_MSUN = 1.0e12

print("正在计算 Fig. 6 单色谱轨迹 ...")
fig6_mass_track_msun, fig6_mono_rate = capture_rate_along_halo_history(
    FIG6_PRESENT_DAY_MASS_MSUN,
    monochromatic_nodes,
    monochromatic_weights,
)
print("正在计算 Fig. 6 log-normal 轨迹 ...")
_, fig6_lognormal_rate = capture_rate_along_halo_history(
    FIG6_PRESENT_DAY_MASS_MSUN,
    lognormal_nodes,
    lognormal_weights,
)

sample_indices = [0, 1, 12, 24, 36, 49]
np.column_stack([
    REDSHIFT[sample_indices],
    fig6_mass_track_msun[sample_indices],
    fig6_mono_rate[sample_indices],
    fig6_lognormal_rate[sample_indices],
])

## 5. 绘制并保存 Fig. 6

左轴是每晕捕获率，右轴是晕质量；横轴和两个纵轴都使用对数尺度。

In [ ]:
figure6, rate_axis6 = plt.subplots(figsize=(7.0, 5.2))
mass_axis6 = rate_axis6.twinx()

rate_axis6.plot(
    REDSHIFT[POSITIVE_REDSHIFT],
    fig6_lognormal_rate[POSITIVE_REDSHIFT],
    color="tab:orange",
    linestyle="--",
    linewidth=2.0,
    label=r"$R_{\rm halo}$ (log-normal)",
)
rate_axis6.plot(
    REDSHIFT[POSITIVE_REDSHIFT],
    fig6_mono_rate[POSITIVE_REDSHIFT],
    color="tab:blue",
    linestyle="-",
    linewidth=2.0,
    label=r"$R_{\rm halo}$ (mono.)",
)
mass_axis6.plot(
    REDSHIFT[POSITIVE_REDSHIFT],
    fig6_mass_track_msun[POSITIVE_REDSHIFT],
    color="tab:red",
    linestyle="-",
    linewidth=1.8,
    label=r"$M_{\rm halo}$",
)

rate_axis6.set_xscale("log")
rate_axis6.set_yscale("log")
mass_axis6.set_yscale("log")
rate_axis6.set_xlim(REDSHIFT[1], 12.0)
rate_axis6.set_ylim(5.0e-13, 1.0e-10)
mass_axis6.set_ylim(1.0e8, 1.0e13)
rate_axis6.set_xlabel(r"Redshift $z$")
rate_axis6.set_ylabel(r"$R_{\rm halo}\;(\mathrm{yr}^{-1})$")
mass_axis6.set_ylabel(r"$M(z)\;[M_\odot]$", color="tab:red")
mass_axis6.tick_params(axis="y", colors="tab:red")
rate_axis6.grid(which="major", color="0.82", linewidth=0.7)
rate_axis6.grid(which="minor", color="0.92", linewidth=0.45)

fig6_lines = rate_axis6.lines + mass_axis6.lines
fig6_labels = [line.get_label() for line in fig6_lines]
rate_axis6.legend(fig6_lines, fig6_labels, loc="lower left", fontsize=8.5)

figure6.tight_layout()
figure6_path = PROJECT_ROOT / "fig6_reproduction.png"
figure6.savefig(figure6_path, dpi=240, bbox_inches="tight")
print(f"Fig. 6 已保存：{figure6_path}")
plt.show()

## 6. 导出 Fig. 6 数据

In [ ]:
fig6_csv_path = PROJECT_ROOT / "fig6_reproduction.csv"
fig6_table = np.column_stack([
    REDSHIFT,
    fig6_mass_track_msun,
    fig6_mono_rate,
    fig6_lognormal_rate,
])
np.savetxt(
    fig6_csv_path,
    fig6_table,
    delimiter=",",
    header=(
        "z,mass_track_msun,"
        "monochromatic_rate_per_year,"
        "lognormal_sigma_0p6_rate_per_year"
    ),
    comments="",
    fmt="%.10e",
)
print(f"Fig. 6 数据已保存：{fig6_csv_path}")

# Fig. 7：不同今天质量晕的单色谱捕获率

## 7. 计算五条 Fig. 7 轨迹

图例中的 $10^3$ 到 $10^{15}M_\odot$ 都是今天 $z=0$ 的质量，而不是每个红移处重新选择的瞬时质量。$M_0=10^{12}M_\odot$ 的单色谱结果直接复用 Fig. 6。

In [ ]:
FIG7_PRESENT_DAY_MASSES_MSUN = np.array([
    1.0e3,
    1.0e6,
    1.0e9,
    1.0e12,
    1.0e15,
])

fig7_mass_tracks = []
fig7_rate_tracks = []

for present_day_mass_msun in FIG7_PRESENT_DAY_MASSES_MSUN:
    if present_day_mass_msun == FIG6_PRESENT_DAY_MASS_MSUN:
        mass_track_msun = fig6_mass_track_msun
        rate_track_per_year = fig6_mono_rate
    else:
        print(
            "正在计算 Fig. 7: "
            f"M0={present_day_mass_msun:.0e} M_sun ..."
        )
        mass_track_msun, rate_track_per_year = (
            capture_rate_along_halo_history(
                present_day_mass_msun,
                monochromatic_nodes,
                monochromatic_weights,
            )
        )

    fig7_mass_tracks.append(mass_track_msun)
    fig7_rate_tracks.append(rate_track_per_year)

fig7_mass_tracks = np.asarray(fig7_mass_tracks)
fig7_rate_tracks = np.asarray(fig7_rate_tracks)
print("Fig. 7 曲线计算完成。")

## 8. 绘制并保存 Fig. 7

In [ ]:
fig7_styles = [
    ("tab:blue", "-"),
    ("tab:orange", "--"),
    ("tab:green", "-."),
    ("tab:red", ":"),
    ("tab:purple", "-"),
]

figure7, axis7 = plt.subplots(figsize=(7.0, 5.2))
for index, present_day_mass_msun in enumerate(FIG7_PRESENT_DAY_MASSES_MSUN):
    exponent = int(np.log10(present_day_mass_msun))
    color, line_style = fig7_styles[index]
    axis7.plot(
        REDSHIFT[POSITIVE_REDSHIFT],
        fig7_rate_tracks[index, POSITIVE_REDSHIFT],
        color=color,
        linestyle=line_style,
        linewidth=2.0,
        label=rf"$M=10^{{{exponent}}}M_\odot$",
    )

axis7.set_xscale("log")
axis7.set_yscale("log")
axis7.set_xlim(REDSHIFT[1], 12.0)
axis7.set_ylim(1.0e-15, 1.0e-10)
axis7.set_xlabel(r"Redshift $z$")
axis7.set_ylabel(r"$R_{\rm halo}\;(\mathrm{yr}^{-1})$")
axis7.legend(loc="lower right", fontsize=8.5)
axis7.grid(which="major", color="0.82", linewidth=0.7)
axis7.grid(which="minor", color="0.92", linewidth=0.45)

figure7.tight_layout()
figure7_path = PROJECT_ROOT / "fig7_reproduction.png"
figure7.savefig(figure7_path, dpi=240, bbox_inches="tight")
print(f"Fig. 7 已保存：{figure7_path}")
plt.show()

## 9. 导出 Fig. 7 数据

CSV 先保存五条实际质量轨迹，再保存对应的五条单色谱捕获率，因此每个 $M_0$ 的环境演化与率可以一一对应。

In [ ]:
fig7_mass_columns = [
    f"mass_track_M0_{mass:.0e}_msun"
    for mass in FIG7_PRESENT_DAY_MASSES_MSUN
]
fig7_rate_columns = [
    f"rate_M0_{mass:.0e}_per_year"
    for mass in FIG7_PRESENT_DAY_MASSES_MSUN
]
fig7_csv_header = ",".join([
    "z",
    *fig7_mass_columns,
    *fig7_rate_columns,
])
fig7_table = np.column_stack([
    REDSHIFT,
    fig7_mass_tracks.T,
    fig7_rate_tracks.T,
])

fig7_csv_path = PROJECT_ROOT / "fig7_reproduction.csv"
np.savetxt(
    fig7_csv_path,
    fig7_table,
    delimiter=",",
    header=fig7_csv_header,
    comments="",
    fmt="%.10e",
)
print(f"Fig. 7 数据已保存：{fig7_csv_path}")

## 10. 当前复现边界

Fig. 7 的 $10^{15}M_\odot$ 曲线继承了项目质量吸积史在高质量端的已知差异：当前 Appendix C 实现在高红移处与论文 Fig. 9 反求轨迹不完全一致。这里保留印刷公式和固定 $M_0$ 身份，不通过排序或未说明的参数微调去消除差异。

Fig. 6 附近的正文称 log-normal 捕获率在 $z<6$ 时比单色谱高约 3 倍，但当前 Appendix A(14) 实现及 Fig. 3-4 得到的是较小且近似恒定的差异。Notebook 保留公式计算结果，不额外乘经验系数去追随这句文字。